# 📐 Fundamentos: Árboles de Decisión

## La idea, en una frase
> **Haz preguntas de sí o no sobre los datos, hasta poder responder con confianza.**

Es lo que hace un mecánico cuando estima el rendimiento de un auto: *¿pesa más de 1,400 kg?* Sí. *¿Tiene más de 6 cilindros?* Sí. Entonces rinde poco. El árbol aprende solo cuáles preguntas hacer y en qué orden.

```
¿displacement > 190?
├── Sí → ¿horsepower > 127?
│         ├── Sí → 14.1 mpg
│         └── No → 19.3 mpg
└── No → ¿model_year > 78?
          ├── Sí → 33.6 mpg
          └── No → 26.5 mpg
```

## El vocabulario
| Término | Qué es |
|---|---|
| **Nodo raíz** | La primera pregunta: la que mejor separa los datos |
| **Nodo interno** | Cada pregunta siguiente |
| **Hoja** | El final del camino: ahí vive la predicción |
| **Profundidad** (`max_depth`) | Cuántas preguntas seguidas puede hacer |

## 1. ¿Cómo elige cada pregunta?

En cada nodo prueba **todos los cortes posibles de todas las variables** y se queda con el que deja los dos grupos resultantes **más parejos por dentro**.

- En **regresión** mide esa "pareja" con el **MSE**: busca que los valores dentro de cada grupo se parezcan entre sí
- En **clasificación** usa el **índice Gini** o la **entropía**: busca que cada grupo quede lo más puro posible, es decir, casi todo de una sola clase

La predicción de una hoja es simplemente el **promedio** de los casos que cayeron ahí (regresión) o la **clase mayoritaria** (clasificación).

## 2. Lo que un árbol no necesita
| | Regresión lineal | KNN | Árbol |
|---|---|---|---|
| ¿Escalar? | No hace falta | **Obligatorio** | **No hace falta** |
| ¿Asume linealidad? | Sí | No | No |
| ¿Se interpreta? | Sí, coeficientes | Poco | **Sí, se dibuja** |

Un árbol no escala porque **no mide distancias**: sólo compara un valor contra un corte. Que el peso esté en libras o en toneladas no cambia nada, porque la pregunta se adapta.

## 3. El problema de los árboles: sobreajustan con facilidad

Si lo dejas crecer sin límite, el árbol seguirá partiendo hasta que **cada hoja tenga un solo caso**. En entrenamiento acierta todo; en datos nuevos, falla. Igual que KNN con k = 1, pero aquí es más fácil de ver.

In [ ]:
import numpy as np
import plotly.express as px
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120)
y = np.sin(x) * 3 + rng.normal(0, 0.6, 120)

datos = {'x': x, 'valor real': y}
for d in (1, 3, None):
    etiqueta = f'max_depth = {d}' if d else 'sin límite'
    datos[etiqueta] = DecisionTreeRegressor(max_depth=d, random_state=42).fit(x.reshape(-1, 1), y).predict(x.reshape(-1, 1))

fig = px.line(datos, x='x', y=[c for c in datos if c != 'x'],
              title='Efecto de max_depth: de escalones gruesos a memorizar el ruido',
              template='plotly_white', labels={'value': 'y', 'variable': ''})
fig.update_traces(selector={'name': 'valor real'}, mode='markers', marker=dict(size=5, color='lightgray'))
fig.update_layout(width=950, height=500)
fig.show()

✅ **Qué observar:**
- Las predicciones son **escalones**, no curvas: dentro de cada hoja el árbol predice siempre el mismo valor
- Con **`max_depth = 1`** hay un solo corte: dos escalones para toda la curva. **Subajuste**
- Con **`max_depth = 3`** la escalera ya sigue la forma de la onda
- **Sin límite**, el árbol pasa por cada punto, incluido el ruido. **Sobreajuste**

Por eso siempre se **poda**: se limita `max_depth` o se exige un mínimo de casos por hoja (`min_samples_leaf`), y el valor se elige con **validación cruzada**.

## 4. Lo que sí regala un árbol: importancia de variables

Cada vez que una variable se usa para cortar, el árbol mide cuánta impureza eliminó. Al sumar esas reducciones se obtiene la **importancia** de cada variable, siempre positiva y sumando 1.

⚠️ Cuidado: a diferencia de un coeficiente, **la importancia no tiene dirección**. Dice que la variable sirve para separar, no si empuja hacia arriba o hacia abajo.

## 5. Lo que hay que recordar
1. El árbol hace **preguntas binarias** y predice el promedio (o la clase mayoritaria) de cada hoja
2. **No necesita escalado** porque no mide distancias
3. **Sobreajusta si no se poda**: `max_depth` y `min_samples_leaf` son el freno
4. Se puede **dibujar y leer**, y da importancia de variables

Ahora llévalo a los dos casos: `caso_mpg.ipynb` y `caso_titanic.ipynb`.